In [1]:
"""
=============================================================================
파트3: 컬리 PB브랜드 마케팅 의사결정 회귀분석
목표변수: 반응량 = 평균감성점수(0~2) × log(리뷰수 + 1)
=============================================================================
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats

plt.rcParams['axes.unicode_minus'] = False
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    try:
        fm.fontManager.addfont(fp)
        plt.rcParams['font.family'] = fm.FontProperties(fname=fp).get_name()
        break
    except Exception:
        pass

# ─────────────────────────────────────────────────────────────────────────────
# OLS 통계량 수동 계산 (statsmodels 대체)
# ─────────────────────────────────────────────────────────────────────────────
def ols_summary(X_df, y_series):
    X = np.column_stack([np.ones(len(y_series)), X_df.values])
    y = y_series.values
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta    = XtX_inv @ X.T @ y
    y_hat   = X @ beta
    resid   = y - y_hat
    ss_res  = resid @ resid
    ss_tot  = ((y - y.mean()) ** 2).sum()
    r2      = 1 - ss_res / ss_tot
    adj_r2  = 1 - (1 - r2) * (n - 1) / (n - k)
    mse     = ss_res / (n - k)
    se      = np.sqrt(np.diag(XtX_inv) * mse)
    t_vals  = beta / se
    p_vals  = 2 * stats.t.sf(np.abs(t_vals), df=n - k)
    f_stat  = (r2 / (k - 1)) / ((1 - r2) / (n - k))
    f_p     = stats.f.sf(f_stat, k - 1, n - k)
    cols    = ['const'] + list(X_df.columns)
    result  = pd.DataFrame({'변수': cols, '계수': beta, '표준오차': se, 't값': t_vals, 'p값': p_vals})
    return result, r2, adj_r2, f_stat, f_p, resid, y_hat

def calc_vif(X_df):
    vif_list = []
    Xv = X_df.values
    for i in range(Xv.shape[1]):
        y_v = Xv[:, i]
        X_v = np.column_stack([np.ones(len(y_v)), np.delete(Xv, i, axis=1)])
        beta_v  = np.linalg.pinv(X_v.T @ X_v) @ X_v.T @ y_v
        y_hat_v = X_v @ beta_v
        ss_res  = ((y_v - y_hat_v) ** 2).sum()
        ss_tot  = ((y_v - y_v.mean()) ** 2).sum()
        r2_v    = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        vif_list.append(1 / (1 - r2_v) if r2_v < 1 else np.inf)
    return pd.DataFrame({'변수': X_df.columns, 'VIF': vif_list})

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. 데이터 로드 및 목표변수 생성
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("파트3: 컬리 PB브랜드 반응량 회귀분석")
print("=" * 70)

df = pd.read_csv('파트2_EDA_통합제품df.csv')
print(f"원본 데이터: {df.shape[0]}행 × {df.shape[1]}열")

df['평균감성점수'] = (df['평균별점'] - 1) / 4 * 2   # 0~2 스케일
df['log_리뷰수']  = np.log1p(df['리뷰수'])
df['반응량']      = df['평균감성점수'] * df['log_리뷰수']

print(f"\n목표변수(반응량) 기술통계")
print(df['반응량'].describe().round(4).to_string())

파트3: 컬리 PB브랜드 반응량 회귀분석
원본 데이터: 976행 × 27열

목표변수(반응량) 기술통계
count    976.0000
mean      13.6920
std        4.2462
min       -0.0000
25%       11.0877
50%       13.9066
75%       16.5293
max       24.9804


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. 독립변수 준비
# ─────────────────────────────────────────────────────────────────────────────
numeric_features = ['할인율','할인가','총용량(g)','단위가격(100g)',
                    '품질순반응','가성비순반응','배송순반응']
df_enc = pd.get_dummies(df, columns=['브랜드','분류'], drop_first=True)
brand_dummies = [c for c in df_enc.columns if c.startswith('브랜드_')]
cat_dummies   = [c for c in df_enc.columns if c.startswith('분류_')]
feature_cols  = numeric_features + brand_dummies + cat_dummies

X_raw = df_enc[feature_cols].copy()
y     = df_enc['반응량'].copy()

scaler = StandardScaler()
X_sc   = X_raw.copy()
X_sc[numeric_features] = scaler.fit_transform(X_raw[numeric_features])

# Convert boolean dummy variables to int to prevent UFuncTypeError in VIF calculation
for col in brand_dummies + cat_dummies:
    if col in X_sc.columns:
        X_sc[col] = X_sc[col].astype(int)

print(f"\n피처 수: {len(feature_cols)}  샘플: {len(y)}")


피처 수: 30  샘플: 976


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. VIF 다중공선성 검사
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 3] VIF 다중공선성 검사")
vif_df = calc_vif(X_sc).sort_values('VIF', ascending=False).reset_index(drop=True)
print(vif_df.head(15).round(2).to_string(index=False))
final_features = vif_df[vif_df['VIF'] <= 10]['변수'].tolist()
print(f"\nVIF ≤ 10 최종 투입 변수: {len(final_features)}개")


[STEP 3] VIF 다중공선성 검사
         변수   VIF
     브랜드_곰곰 58.22
  브랜드_KF365 46.10
브랜드_kurly's 16.61
      배송순반응  8.13
     가성비순반응  8.05
    브랜드_차려낸  7.47
      품질순반응  3.48
  브랜드_마이퍼스트  2.43
    분류_메인요리  1.90
     총용량(g)  1.75
      분류_정육  1.63
    분류_베이커리  1.60
        할인가  1.57
       분류_쌀  1.52
        할인율  1.49

VIF ≤ 10 최종 투입 변수: 27개


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. OLS 회귀분석
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 4] OLS 다중회귀분석")
X_final = X_sc[final_features]
coef_df, r2, adj_r2, f_stat, f_p, resid, y_hat = ols_summary(X_final, y)
coef_df['절대계수'] = coef_df['계수'].abs()
coef_df['유의']    = coef_df['p값'] < 0.05

print(f"R²={r2:.4f}  Adj.R²={adj_r2:.4f}  F-p={f_p:.2e}  N={len(y)}")
print("\n전체 회귀계수 (|계수| 내림차순):")
tbl = coef_df[coef_df['변수']!='const'].sort_values('절대계수', ascending=False)
print(tbl[['변수','계수','표준오차','t값','p값','유의']].round(4).to_string(index=False))


[STEP 4] OLS 다중회귀분석
R²=0.6391  Adj.R²=0.6289  F-p=1.13e-188  N=976

전체 회귀계수 (|계수| 내림차순):
        변수      계수   표준오차       t값     p값    유의
    가성비순반응  5.2890 0.2273  23.2667 0.0000  True
     품질순반응 -3.0594 0.1252 -24.4430 0.0000  True
     배송순반응 -1.7374 0.2347  -7.4014 0.0000  True
 브랜드_마이퍼스트 -1.1954 1.2091  -0.9887 0.3231 False
    분류_간편식  1.1879 0.4596   2.5847 0.0099  True
     분류_두부  1.0798 0.4935   2.1882 0.0289  True
     분류_어묵 -1.0697 0.6147  -1.7402 0.0822 False
     분류_육수 -0.9953 0.5177  -1.9227 0.0548 False
   분류_메인요리 -0.9589 1.1807  -0.8121 0.4169 False
       할인가 -0.9292 0.1033  -8.9931 0.0000  True
      분류_쌀 -0.8187 0.5769  -1.4191 0.1562 False
     분류_정육 -0.6520 0.3507  -1.8590 0.0633 False
  분류_과일·채소  0.6356 0.3510   1.8111 0.0704 False
     분류_해산  0.6179 0.9380   0.6587 0.5103 False
     분류_수산  0.6118 0.3856   1.5865 0.1130 False
    분류_샐러드 -0.5667 0.4811  -1.1779 0.2391 False
     분류_달걀  0.4599 0.3710   1.2396 0.2154 False
    분류_건어물 -0.3704 0.5174  -0.7159 0.4742 Fals

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. 유의 변수 추출
# ─────────────────────────────────────────────────────────────────────────────
sig_df  = coef_df[(coef_df['유의']) & (coef_df['변수']!='const')]
sig_pos = sig_df[sig_df['계수']>0].sort_values('절대계수', ascending=False)
sig_neg = sig_df[sig_df['계수']<0].sort_values('절대계수', ascending=False)

print("\n반응량 증가 요인 (양의 유의 변수):")
print(sig_pos[['변수','계수','p값']].round(4).to_string(index=False))
print("\n반응량 감소 요인 (음의 유의 변수):")
print(sig_neg[['변수','계수','p값']].round(4).to_string(index=False))


반응량 증가 요인 (양의 유의 변수):
    변수     계수     p값
가성비순반응 5.2890 0.0000
분류_간편식 1.1879 0.0099
 분류_두부 1.0798 0.0289
총용량(g) 0.2818 0.0098

반응량 감소 요인 (음의 유의 변수):
   변수      계수  p값
품질순반응 -3.0594 0.0
배송순반응 -1.7374 0.0
  할인가 -0.9292 0.0


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. 연구 질문 검증
# ─────────────────────────────────────────────────────────────────────────────
def get_coef(var):
    row = coef_df[coef_df['변수']==var]
    return (row['계수'].values[0], row['p값'].values[0]) if len(row)>0 else (None,None)

print("\n[STEP 6] 연구 질문 검증")

print("\nQ1: 가격(할인가) vs 품질순반응 → 반응량 상쇄 여부")
for var in ['할인가','품질순반응']:
    c, p = get_coef(var)
    mark = "★유의" if p is not None and p<0.05 else "비유의"
    print(f"  {var:12s}  계수={c:+.4f}  p={p:.4f}  [{mark}]")
c_p, _ = get_coef('할인가'); c_q, _ = get_coef('품질순반응')
if c_p is not None and c_q is not None:
    if c_q > 0 and abs(c_q) > abs(c_p):
        print("  → 결론: 품질순반응 효과가 가격 효과보다 크다. 품질 강화로 가격 부담 일부 상쇄 가능.")
    elif c_q > 0:
        print("  → 결론: 품질순반응은 양의 효과이나 가격 부담을 완전히 상쇄하지 못함.")

print("\nQ2: 배송순반응 → 반응량 영향")
c_d, p_d = get_coef('배송순반응')
if c_d is not None:
    mark = "★유의" if p_d<0.05 else "비유의"
    print(f"  배송순반응  계수={c_d:+.4f}  p={p_d:.4f}  [{mark}]")
    if c_d>0 and p_d<0.05:
        print("  → 결론: 배송 긍정↑ → 반응량 유의 상승. 배송 부정은 직접적인 반응량 감소 요인.")
    else:
        print("  → 결론: 다른 변수 통제 시 배송 효과 통계적으로 미약. 품질/가성비 효과가 더 지배적.")

print("\nQ3: 분류별 반응량 (낮은 순)")
cat_stats = df.groupby('분류')['반응량'].agg(['mean','median','count']).round(3)
cat_stats.columns = ['평균반응량','중앙값','상품수']
print(cat_stats.sort_values('평균반응량').to_string())

cat_coef_rows = coef_df[coef_df['변수'].str.startswith('분류_')].sort_values('계수')
if len(cat_coef_rows)>0:
    print("\n분류 더미 회귀계수 (기준범주 대비, ★=유의):")
    for _, row in cat_coef_rows.iterrows():
        mark = " ★" if row['유의'] else ""
        print(f"  {row['변수']:20s}  계수={row['계수']:+.4f}  p={row['p값']:.4f}{mark}")

brand_coef_rows = coef_df[coef_df['변수'].str.startswith('브랜드_')].sort_values('계수')
if len(brand_coef_rows)>0:
    print("\n브랜드 더미 회귀계수 (기준브랜드 대비, ★=유의):")
    for _, row in brand_coef_rows.iterrows():
        mark = " ★" if row['유의'] else ""
        print(f"  {row['변수']:20s}  계수={row['계수']:+.4f}  p={row['p값']:.4f}{mark}")


[STEP 6] 연구 질문 검증

Q1: 가격(할인가) vs 품질순반응 → 반응량 상쇄 여부
  할인가           계수=-0.9292  p=0.0000  [★유의]
  품질순반응         계수=-3.0594  p=0.0000  [★유의]

Q2: 배송순반응 → 반응량 영향
  배송순반응  계수=-1.7374  p=0.0000  [★유의]
  → 결론: 다른 변수 통제 시 배송 효과 통계적으로 미약. 품질/가성비 효과가 더 지배적.

Q3: 분류별 반응량 (낮은 순)
        평균반응량     중앙값  상품수
분류                        
건어물    11.983  11.954   30
육수     12.214  11.992   30
어묵     12.384  13.532   21
가공육    12.406  12.130  189
해산     12.482  11.976    8
쌀      13.016  13.306   31
메인요리   13.034  15.679    5
정육     13.051  14.623   96
밀키트    13.055  13.194   38
국      13.728  13.054   42
견과     14.000  14.498   33
수산     14.008  13.954   64
달걀     14.160  14.159   73
과일·채소  14.164  14.678   86
베이커리   14.690  15.119   70
반찬     14.945  15.288   45
샐러드    15.117  15.503   37
간편식    15.789  16.011   43
두부     16.835  18.039   35

분류 더미 회귀계수 (기준범주 대비, ★=유의):
  분류_어묵                 계수=-1.0697  p=0.0822
  분류_육수                 계수=-0.9953  p=0.0548
  분류_메인요리               계수=-0.9589  p=0.416

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Ridge / Lasso 비교
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 7] 모델 안정성 비교 (5-Fold CV)")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_final, y, test_size=0.2, random_state=42)

model_results = {}
for name, model in [('OLS', LinearRegression()),
                    ('Ridge(α=1)', Ridge(alpha=1)),
                    ('Lasso(α=0.01)', Lasso(alpha=0.01, max_iter=10000))]:
    cv = cross_val_score(model, X_final, y, cv=kf, scoring='r2')
    model.fit(X_tr, y_tr)
    tr2  = r2_score(y_te, model.predict(X_te))
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    model_results[name] = {'CV R²(mean)': cv.mean(), 'CV R²(std)': cv.std(),
                           'Test R²': tr2, 'RMSE': rmse}
    print(f"  {name:22s} | CV R²={cv.mean():.4f}(±{cv.std():.4f}) | Test R²={tr2:.4f} | RMSE={rmse:.4f}")



[STEP 7] 모델 안정성 비교 (5-Fold CV)
  OLS                    | CV R²=0.6060(±0.0262) | Test R²=0.5853 | RMSE=2.7367
  Ridge(α=1)             | CV R²=0.6091(±0.0249) | Test R²=0.5854 | RMSE=2.7364
  Lasso(α=0.01)          | CV R²=0.6141(±0.0267) | Test R²=0.5842 | RMSE=2.7405


In [15]:
# [한글폰트 설치 - Colab]
import pandas as pd
import numpy as np

# Install Korean font
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# Configure Matplotlib to use NanumGothic
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False # To prevent breaking minus sign

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 7 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (25.7 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 118194 files and direct

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. 시각화
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 8] 시각화 생성 중...")
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle('컬리 PB브랜드 파트3: 반응량 회귀분석', fontsize=15, fontweight='bold')

# 패널1: 유의 변수 계수
ax1 = axes[0, 0]
sig_plot = coef_df[(coef_df['유의']) & (coef_df['변수']!='const')].sort_values('계수')
colors = ['#e74c3c' if c<0 else '#27ae60' for c in sig_plot['계수']]
ax1.barh(sig_plot['변수'], sig_plot['계수'], color=colors, edgecolor='white', height=0.7)
ax1.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_title('유의 변수 표준화 회귀계수 (p<0.05)', fontsize=11)
ax1.set_xlabel('표준화 계수')
ax1.tick_params(axis='y', labelsize=8)

# 패널2: 잔차 플롯
ax2 = axes[0, 1]
ax2.scatter(y_hat, resid, alpha=0.35, s=12, color='#2980b9')
ax2.axhline(0, color='red', linewidth=1, linestyle='--')
ax2.set_title('잔차 플롯 (Fitted vs Residuals)', fontsize=11)
ax2.set_xlabel('예측값(반응량)')
ax2.set_ylabel('잔차')

# 패널3: 분류별 평균 반응량
ax3 = axes[1, 0]
cat_mean = df.groupby('분류')['반응량'].mean().sort_values()
med_val  = cat_mean.median()
clr3 = ['#e74c3c' if v<med_val else '#27ae60' for v in cat_mean]
cat_mean.plot(kind='barh', ax=ax3, color=clr3, edgecolor='white')
ax3.axvline(med_val, color='navy', lw=1.2, ls='--', label=f'중앙값={med_val:.2f}')
ax3.set_title('분류별 평균 반응량 (빨강=중앙값 미만)', fontsize=11)
ax3.set_xlabel('평균 반응량')
ax3.legend(fontsize=8)
ax3.tick_params(axis='y', labelsize=9)

# 패널4: 브랜드별 박스플롯
ax4 = axes[1, 1]
brand_order = df.groupby('브랜드')['반응량'].median().sort_values().index.tolist()
brand_data  = [df[df['브랜드']==b]['반응량'].values for b in brand_order]
palette4 = ['#3498db','#2ecc71','#e74c3c','#f39c12','#9b59b6','#1abc9c']
bp = ax4.boxplot(brand_data, labels=brand_order, patch_artist=True,
                 medianprops={'color':'red','linewidth':2},
                 flierprops={'markersize':3,'alpha':0.4})
for patch, color in zip(bp['boxes'], palette4[:len(brand_order)]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax4.set_title('브랜드별 반응량 분포', fontsize=11)
ax4.set_ylabel('반응량')
ax4.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('part3_regression_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print("  → part3_regression_plots.png 저장 완료")


[STEP 8] 시각화 생성 중...
  → part3_regression_plots.png 저장 완료


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. 마케팅 의사결정 요약
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("마케팅 의사결정 요약 리포트")
print("="*70)
print(f"\n모델: R²={r2:.4f}  Adj.R²={adj_r2:.4f}  F-p={f_p:.2e}  N={len(y)}")
print("\n▶ 반응량 TOP 증가 요인 (표준화 계수 기준):")
for _, row in sig_pos.head(5).iterrows():
    print(f"  +{row['변수']:25s}  계수={row['계수']:+.4f}  p={row['p값']:.4f}")
print("\n▶ 반응량 TOP 감소 요인:")
for _, row in sig_neg.head(5).iterrows():
    print(f"  -{row['변수']:25s}  계수={row['계수']:+.4f}  p={row['p값']:.4f}")
print("\n▶ 분류별 하위 3개 (개선 우선순위):")
for idx, row in cat_stats.sort_values('평균반응량').head(3).iterrows():
    print(f"  [{idx}]  평균반응량={row['평균반응량']:.3f}  상품수={int(row['상품수'])}개")
print("\n▶ 모델 안정성:")
for name, res in model_results.items():
    print(f"  {name:22s}  CV R²={res['CV R²(mean)']:.4f}  Test R²={res['Test R²']:.4f}  RMSE={res['RMSE']:.4f}")
print("\n완료. 출력: part3_regression_plots.png  /  part3_regression_analysis.py")



마케팅 의사결정 요약 리포트

모델: R²=0.6391  Adj.R²=0.6289  F-p=1.13e-188  N=976

▶ 반응량 TOP 증가 요인 (표준화 계수 기준):
  +가성비순반응                     계수=+5.2890  p=0.0000
  +분류_간편식                     계수=+1.1879  p=0.0099
  +분류_두부                      계수=+1.0798  p=0.0289
  +총용량(g)                     계수=+0.2818  p=0.0098

▶ 반응량 TOP 감소 요인:
  -품질순반응                      계수=-3.0594  p=0.0000
  -배송순반응                      계수=-1.7374  p=0.0000
  -할인가                        계수=-0.9292  p=0.0000

▶ 분류별 하위 3개 (개선 우선순위):
  [건어물]  평균반응량=11.983  상품수=30개
  [육수]  평균반응량=12.214  상품수=30개
  [어묵]  평균반응량=12.384  상품수=21개

▶ 모델 안정성:
  OLS                     CV R²=0.6060  Test R²=0.5853  RMSE=2.7367
  Ridge(α=1)              CV R²=0.6091  Test R²=0.5854  RMSE=2.7364
  Lasso(α=0.01)           CV R²=0.6141  Test R²=0.5842  RMSE=2.7405

완료. 출력: part3_regression_plots.png  /  part3_regression_analysis.py
